# Backpropagation

**Goal:** Implement forward and manual backward passes for a 2-layer MLP from scratch in PyTorch.
Derive and code the exact gradients dW1, db1, dW2, db2 using the clean `softmax − one-hot` output gradient.
Validate every manual gradient against `torch.autograd` with `torch.allclose`, then show the idiomatic `.backward()` equivalent.

Backpropagation is reverse-mode automatic differentiation: it walks the computation graph backward, reusing cached intermediate activations so the full gradient costs only a small constant factor more than the forward pass.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # noqa
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)


running on: mps


## Network Architecture and Data

We build a 2-layer MLP for multi-class classification:

```
z1 = X @ W1 + b1        # (B, Din) -> (B, H)  pre-activations layer 1
a1 = relu(z1)            # (B, H)              hidden activations
z2 = a1 @ W2 + b2        # (B, H)  -> (B, C)  logits
loss = softmax-cross-entropy(z2, y)             scalar
```

Dimensions: B=64 samples, Din=8 input features, H=16 hidden units, C=4 classes.

We use **float64 on CPU** for the manual-gradient check so that numerical precision does not mask errors.

In [2]:
# Use cpu + float64 for the manual gradient verification (avoids MPS float32 rounding)
# Production use would stay on the configured device/dtype.
cpu64 = torch.device("cpu")
dtype64 = torch.float64

torch.manual_seed(42)

B, Din, H, C = 64, 8, 16, 4

# Random input and integer class labels
X = torch.randn(B, Din, dtype=dtype64, device=cpu64)
y = torch.randint(0, C, (B,), device=cpu64)  # shape (B,)

# Parameters — we will compare manual grads against autograd, so keep two copies
W1 = torch.randn(Din, H, dtype=dtype64, device=cpu64) * 0.01
b1 = torch.zeros(H, dtype=dtype64, device=cpu64)
W2 = torch.randn(H, C, dtype=dtype64, device=cpu64) * 0.01
b2 = torch.zeros(C, dtype=dtype64, device=cpu64)

print(f"X  shape: {X.shape}")
print(f"y  shape: {y.shape}")
print(f"W1 shape: {W1.shape}")
print(f"b1 shape: {b1.shape}")
print(f"W2 shape: {W2.shape}")
print(f"b2 shape: {b2.shape}")


X  shape: torch.Size([64, 8])
y  shape: torch.Size([64])
W1 shape: torch.Size([8, 16])
b1 shape: torch.Size([16])
W2 shape: torch.Size([16, 4])
b2 shape: torch.Size([4])


## Manual Forward Pass

The forward pass runs left-to-right through the computation graph, caching every intermediate value needed for the backward pass.

Key caches:
- `z1` — needed for the ReLU mask (which units were active)
- `a1` — needed to compute `dW2 = a1.T @ dz2`
- `z2` (logits) — needed to compute the softmax probabilities


In [3]:
def relu(z: torch.Tensor) -> torch.Tensor:
    return z.clamp(min=0)


def softmax(z: torch.Tensor) -> torch.Tensor:
    """Numerically stable softmax along last dimension."""
    z_shift = z - z.max(dim=-1, keepdim=True).values
    exp_z = torch.exp(z_shift)
    return exp_z / exp_z.sum(dim=-1, keepdim=True)


def cross_entropy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Stable softmax cross-entropy: -z_c + logsumexp(z)."""
    z_shift = logits - logits.max(dim=-1, keepdim=True).values
    log_sum_exp = torch.log(torch.exp(z_shift).sum(dim=-1))  # (B,)
    log_p_correct = z_shift[torch.arange(len(targets)), targets]  # (B,)
    return (-log_p_correct + log_sum_exp).mean()


def forward_pass(
    X: torch.Tensor,
    W1: torch.Tensor,
    b1: torch.Tensor,
    W2: torch.Tensor,
    b2: torch.Tensor,
    y: torch.Tensor,
) -> tuple[torch.Tensor, dict]:
    """Run forward pass; return (scalar_loss, cache)."""
    z1 = X @ W1 + b1          # (B, H)
    a1 = relu(z1)              # (B, H) — cache z1 for ReLU mask
    z2 = a1 @ W2 + b2          # (B, C)
    loss = cross_entropy_from_logits(z2, y)

    cache = {"X": X, "z1": z1, "a1": a1, "z2": z2, "W2": W2, "y": y}
    return loss, cache


loss_manual, cache = forward_pass(X, W1, b1, W2, b2, y)
print(f"Forward pass loss: {loss_manual.item():.6f}")


Forward pass loss: 1.386209


## Manual Backward Pass — Deriving the Gradients

We derive each gradient using the chain rule, working backward from the loss.

**Output layer gradient (clean form):**

The gradient of softmax cross-entropy with respect to logits `z2` is:
```
dz2 = (softmax(z2) - one_hot(y)) / B
```
This is the "clean" form: the softmax and cross-entropy gradients cancel elegantly.

**Then, propagating backward:**
```
dW2 = a1.T @ dz2          # (H, C)
db2 = dz2.sum(axis=0)     # (C,)

da1 = dz2 @ W2.T          # (B, H)  — gradient flows through W2 transposed
dz1 = da1 * relu'(z1)     # (B, H)  — ReLU mask: 1 where z1 > 0, else 0

dW1 = X.T @ dz1            # (Din, H)
db1 = dz1.sum(axis=0)     # (H,)
```

**Why reverse-mode reuses intermediates:**
Each backward step uses cached forward values (a1, z1, X) plus the upstream gradient.
No forward quantity is recomputed; the full gradient over all parameters costs O(1) times the forward pass cost.


In [4]:
def backward_pass(cache: dict) -> dict[str, torch.Tensor]:
    """Compute all parameter gradients manually using cached forward values."""
    X, z1, a1, z2, W2, y = (
        cache["X"], cache["z1"], cache["a1"],
        cache["z2"], cache["W2"], cache["y"],
    )
    B = X.shape[0]

    # --- Output layer gradient ---
    # dL/dz2: shape (B, C)
    # softmax(z2) - one_hot(y), then divide by B (mean reduction)
    probs = softmax(z2)                              # (B, C)
    dz2 = probs.clone()                              # copy; do NOT mutate probs
    dz2[torch.arange(B), y] -= 1.0                  # subtract 1 at correct class
    dz2 = dz2 / B                                   # (B, C) — mean reduction

    # --- Layer 2 parameter gradients ---
    dW2 = a1.T @ dz2                                 # (H, C)
    db2 = dz2.sum(dim=0)                             # (C,)

    # --- Gradient flowing back through W2 ---
    da1 = dz2 @ W2.T                                 # (B, H)

    # --- ReLU backward: zero out gradient where preactivation was <= 0 ---
    drelu_mask = (z1 > 0).to(dtype=z1.dtype)        # (B, H) — 1 where active
    dz1 = da1 * drelu_mask                           # (B, H)

    # --- Layer 1 parameter gradients ---
    dW1 = X.T @ dz1                                  # (Din, H)
    db1 = dz1.sum(dim=0)                             # (H,)

    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}


manual_grads = backward_pass(cache)

print("Manual gradient shapes:")
for name, g in manual_grads.items():
    print(f"  {name}: {g.shape}")


Manual gradient shapes:
  dW1: torch.Size([8, 16])
  db1: torch.Size([16])
  dW2: torch.Size([16, 4])
  db2: torch.Size([4])


## Validation: Manual Gradients vs. `torch.autograd`

We run the identical forward pass with `requires_grad=True` parameters, call `.backward()`, and compare every gradient with `torch.allclose(atol=1e-5)`.


In [5]:
# Autograd computation — identical parameters, requires_grad=True
W1_ag = W1.clone().requires_grad_(True)
b1_ag = b1.clone().requires_grad_(True)
W2_ag = W2.clone().requires_grad_(True)
b2_ag = b2.clone().requires_grad_(True)

# Forward pass (identical arithmetic)
z1_ag = X @ W1_ag + b1_ag
a1_ag = relu(z1_ag)
z2_ag = a1_ag @ W2_ag + b2_ag
loss_ag = cross_entropy_from_logits(z2_ag, y)

loss_ag.backward()

print("Loss comparison:")
print(f"  Manual forward loss : {loss_manual.item():.8f}")
print(f"  Autograd loss       : {loss_ag.item():.8f}")

# Compare each gradient
pairs = [
    ("dW1", manual_grads["dW1"], W1_ag.grad),
    ("db1", manual_grads["db1"], b1_ag.grad),
    ("dW2", manual_grads["dW2"], W2_ag.grad),
    ("db2", manual_grads["db2"], b2_ag.grad),
]

print("\nGradient comparison (manual vs. autograd):")
for name, man, auto in pairs:
    max_err = (man - auto).abs().max().item()
    match = torch.allclose(man, auto, atol=1e-5)
    print(f"  {name}: max|err|={max_err:.2e}  allclose={match}")

# Hard assertions — these must not be weakened
assert torch.allclose(manual_grads["dW1"], W1_ag.grad, atol=1e-5), "dW1 mismatch"
assert torch.allclose(manual_grads["db1"], b1_ag.grad, atol=1e-5), "db1 mismatch"
assert torch.allclose(manual_grads["dW2"], W2_ag.grad, atol=1e-5), "dW2 mismatch"
assert torch.allclose(manual_grads["db2"], b2_ag.grad, atol=1e-5), "db2 mismatch"

print("\nAll assertions passed: manual backward matches autograd exactly.")


Loss comparison:
  Manual forward loss : 1.38620900
  Autograd loss       : 1.38620900

Gradient comparison (manual vs. autograd):
  dW1: max|err|=4.34e-19  allclose=True
  db1: max|err|=4.34e-19  allclose=True
  dW2: max|err|=8.67e-19  allclose=True
  db2: max|err|=2.78e-17  allclose=True

All assertions passed: manual backward matches autograd exactly.


## Why Reverse-Mode Reuses Intermediates

Forward-mode differentiation tracks how a small perturbation in one input propagates forward.
It is efficient when there are few inputs and many outputs.

**Reverse-mode** (backprop) starts from the scalar loss and propagates sensitivity backward.
It computes all parameter gradients in a single backward sweep because:

1. Each cached intermediate (a1, z1, X) is reused by exactly one backward step.
2. No value is recomputed — the backward pass reads the cache written during the forward pass.
3. Total backward cost ≈ 2–3× the forward cost, regardless of the number of parameters.

Without reverse-mode, we would need two forward passes per parameter (one perturbed up, one down) for central finite differences to estimate gradients — completely infeasible for millions of parameters.

## Idiomatic PyTorch

In practice, `torch.nn` modules handle the manual bookkeeping automatically.
`nn.Linear` stores W and b; `F.cross_entropy` combines softmax and NLL; `.backward()` walks the computation graph.

In [6]:
import torch.nn as nn

torch.manual_seed(42)

# Build the same 2-layer MLP with nn.Linear
model = nn.Sequential(
    nn.Linear(Din, H),
    nn.ReLU(),
    nn.Linear(H, C),
).to(device=device, dtype=torch.float32)  # use configured device for training

# Move data to configured device (float32 for production)
X_dev = X.to(device=device, dtype=torch.float32)
y_dev = y.to(device=device)

# Forward + backward in three lines
logits = model(X_dev)
loss_nn = F.cross_entropy(logits, y_dev)
loss_nn.backward()

print(f"nn.Sequential loss: {loss_nn.item():.6f}")
print(f"Grad norm W1 (nn.Linear): {model[0].weight.grad.norm().item():.4f}")
print(f"Grad norm W2 (nn.Linear): {model[2].weight.grad.norm().item():.4f}")
print("\nNote: nn.Linear stores W as shape (out, in), so weight.T matches our W convention.")


nn.Sequential loss: 1.422379
Grad norm W1 (nn.Linear): 0.1363
Grad norm W2 (nn.Linear): 0.2532

Note: nn.Linear stores W as shape (out, in), so weight.T matches our W convention.


## Takeaways

- **Backpropagation** is reverse-mode automatic differentiation applied to neural networks: it chains local vector-Jacobian products backward through the computation graph.
- **Forward pass caches** intermediates (`z1`, `a1`, `z2`); the backward pass reads them — no recomputation needed.
- **Softmax cross-entropy gradient** is the clean expression `(softmax(z2) − one_hot(y)) / B`, arising because the softmax Jacobian and cross-entropy gradient combine analytically.
- **ReLU backward** is a binary mask: gradient passes through where the preactivation was positive, and is zeroed where it was negative.
- **Reverse-mode is efficient** for training: one backward sweep computes all gradients, costing ≈2–3× the forward pass regardless of parameter count. Finite differences would cost 2P forward passes for P parameters.
- **In-place mutation** corrupts cached values needed by the backward pass — always create new tensors.
- See also: `[[chain-rule]]`, `[[gradients-and-jacobians]]`, `[[activations-tanh-relu]]`, `[[vanishing-exploding-gradients]]`
